[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusicalInformatics/miws26/blob/main/introduction/intro_to_fourier_analysis.ipynb)

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    import os

    # Install python packages
    ! pip install git+https://github.com/CPJKU/partitura.git@develop ipympl

    # To be able to access helper modules and data in the repo for this tutorial
    # (not necessary if the jupyter notebook is run locally instead of google colab)
    if not os.path.exists("./miws26"):
        !git clone --recurse-submodules https://github.com/MusicalInformatics/miws26.git
    import sys
    sys.path.insert(0, "./miws26/introduction/")

    # Colab has already imported matplotlib before ipympl is installed, which
    # breaks the interactive backend (%matplotlib widget). Restart the runtime
    # once after the first installation; afterwards just run the notebook again.
    if not os.path.exists("/content/.ipympl_ready"):
        open("/content/.ipympl_ready", "w").close()
        print("Packages installed. Restarting the runtime: please run this cell again.")
        os.kill(os.getpid(), 9)

    # Needed for interactive matplotlib figures (%matplotlib widget) on Colab
    from google.colab import output
    output.enable_custom_widget_manager()

# Introduction to Fourier Analysis

This notebook presents a very short (and not very formal!) introduction to Fourier analysis.

Music signals can be fairly complex. Because of this, we would like to be able to extract relevant information. This is especially true for audio. One of the most useful representations, is representing information in terms of sinusoids.

But before going there, let's start with a basic mathematical question: how do we compute the similarity between two functions? (In here, imagine that we want to represent audio signals in terms of mathematical functions over time).

Imagine that we have real valued functions $f:\mathbb{R} \mapsto \mathbb{R}$ and $g:\mathbb{R} \mapsto \mathbb{R}$.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

## 1. Comparing signals: the inner product

One way to measure how similar two signals are is to multiply them and add up the result over time, i.e., to compute their **inner product**. The larger the inner product, the more similar the signals:

$$\langle f, g \rangle = \int_{t_0}^{T} f(t)\, g(t)\, dt$$

Signals are just functions of time, so we can define them as Python functions. We approximate the integral numerically (with the trapezoidal rule).

To build some intuition, we will plot the two signals $f$ and $g$ (top) and their **product** $f(t)\, g(t)$ (bottom). The inner product is the *signed area* under the product: the green areas (where the product is positive) count positively, the red areas (where it is negative) count negatively.

- If $f$ and $g$ go up and down together, the product is mostly positive and the inner product is large.
- If one goes up when the other goes down, the product is mostly negative.
- If they are unrelated, the positive and negative areas cancel out and the inner product is (close to) zero.

In [ ]:
def inner_product(f, g, t0=0.0, T=1.0, n_points=10000):
    """Inner product of two signals (functions of t) between t0 and T.
    (for complex signals g is conjugated, which does nothing for real signals)"""
    t = np.linspace(t0, T, n_points)
    product = np.broadcast_to(f(t) * np.conj(g(t)), t.shape)  # also works for constants
    return np.trapezoid(product, t)


def plot_inner_products(examples, t0=0.0, T=1.0):
    """Plot signals f and g (top) and their product (bottom), for a list of
    examples [(f, g, title), ...]. The inner product is the signed area
    under the product."""
    t = np.linspace(t0, T, 1000)
    n = len(examples)
    fig, axes = plt.subplots(2, n, figsize=(max(4 * n, 6), 5), sharex=True, squeeze=False)
    for j, (f, g, title) in enumerate(examples):
        ft = np.broadcast_to(f(t), t.shape)
        gt = np.broadcast_to(g(t), t.shape)
        product = ft * gt

        ax = axes[0, j]
        ax.plot(t, ft, color="navy", label="f")
        ax.plot(t, gt, color="darkorange", label="g")
        ax.axhline(0, color="gray", linewidth=0.5)
        ax.set_title(title)
        ax.legend(loc="upper left")

        ax = axes[1, j]
        ax.plot(t, product, color="black", linewidth=1)
        ax.fill_between(t, product, 0, where=product >= 0, color="tab:green", alpha=0.5, interpolate=True)
        ax.fill_between(t, product, 0, where=product < 0, color="tab:red", alpha=0.5, interpolate=True)
        ax.axhline(0, color="gray", linewidth=0.5)
        ax.set_xlabel("t")
        ax.set_title(f"<f, g> = {inner_product(f, g, t0, T).real:.2f}", fontsize=10)
    axes[0, 0].set_ylabel("signals")
    axes[1, 0].set_ylabel("f(t) g(t)")
    plt.tight_layout()
    plt.show()

### Lines

Let's start with the simplest signals: straight lines (and constants) between $t_0 = -1$ and $T = 1$.

In [ ]:
def f(t):
    return t


def g_same(t):
    return t  # the same line


def g_opposite(t):
    return -t  # the opposite line


def g_constant(t):
    return np.ones_like(t)  # a constant signal


plot_inner_products(
    [
        (f, g_same, "same line"),
        (f, g_opposite, "opposite line"),
        (f, g_constant, "line vs. constant"),
    ],
    t0=-1,
    T=1,
)

- Same line: the product $t \cdot t = t^2$ is never negative, so the inner product is positive.
- Opposite line: the product is never positive, so the inner product is negative.
- Line vs. constant: the positive and negative areas are exactly the same size, so they cancel out. The inner product is zero, and we say that the signals are **orthogonal**.

### Polynomials

The same idea works with any other signal, for example polynomials. Which of these do you expect to be similar?

In [ ]:
def f(t):
    return t


def g1(t):
    return t**3


def g2(t):
    return t**2


def g3(t):
    return 3 * t**2 - 1


plot_inner_products(
    [
        (f, g1, "t vs. t^3"),
        (f, g2, "t vs. t^2"),
        (g_constant, g3, "1 vs. 3t^2 - 1"),
    ],
    t0=-1,
    T=1,
)

- $t$ and $t^3$ have a similar shape (both are odd, and go up from left to right), so they are similar.
- $t$ and $t^2$ are not similar at all. $t^2$ is symmetric around 0, and $t$ is antisymmetric, so the areas cancel out and they are orthogonal.
- $1$ and $3t^2 - 1$ do not look alike or have any obvious symmetry, but they are also orthogonal!

### Play with it

Choose a reference signal $f$ and define $g(t) = c_0 + c_1 t + c_2 t^2 + c_3 t^3$ with the sliders. Can you find a $g$ with a large positive inner product with $f$? What about a large negative one? Can you find a $g$ that is orthogonal to $f$ (an inner product of 0), without using $g = 0$?

In [ ]:
from ipywidgets import interact

references = {
    "1": lambda t: np.ones_like(t),
    "t": lambda t: t,
    "t^2": lambda t: t**2,
    "t^3": lambda t: t**3,
}


def explore_inner_product(reference="t", c0=0.0, c1=1.0, c2=0.0, c3=0.0):
    f = references[reference]

    def g(t):
        return c0 + c1 * t + c2 * t**2 + c3 * t**3

    plot_inner_products([(f, g, f"f(t) = {reference}")], t0=-1, T=1)


interact(
    explore_inner_product,
    reference=list(references),
    c0=(-2.0, 2.0, 0.1),
    c1=(-2.0, 2.0, 0.1),
    c2=(-2.0, 2.0, 0.1),
    c3=(-2.0, 2.0, 0.1),
);

### Sinusoids

Now let's compare sinusoids, which will be our main building blocks. Here we compare a 3 Hz sine $f$ (on the interval from 0 to 1 second) with different signals. **Try changing the signals** (the frequencies, the amplitudes, the interval $[t_0, T]$) and see how the inner product changes!

In [ ]:
def f(t):
    return np.sin(2 * np.pi * 3 * t)  # 3 Hz sine


def g(t):
    return np.sin(2 * np.pi * 3 * t)  # the same signal


def h(t):
    return np.sin(2 * np.pi * 4 * t)  # 4 Hz sine


def k(t):
    return np.cos(2 * np.pi * 3 * t)  # 3 Hz cosine (a sine shifted by a quarter period)


plot_inner_products(
    [
        (f, g, "same signal"),
        (f, h, "different frequency"),
        (f, k, "shifted by 90 degrees"),
    ],
    t0=0,
    T=1,
)

The inner product of a signal with itself is large (the signals are perfectly aligned), while sines with different frequencies, or a sine and a cosine of the same frequency, have an inner product of (almost) zero: the positive and negative areas cancel out and the signals are **orthogonal**.

## 2. The Fourier basis: sines and cosines

Sines and cosines whose frequencies are integer multiples of $1/T$ are all orthogonal to each other on an interval of length $T$:

$$\cos\left(\frac{2\pi n t}{T}\right), \quad \sin\left(\frac{2\pi n t}{T}\right), \quad n = 1, 2, 3, \dots$$

We can check this by computing the inner products between all pairs of these functions (with $T = 1$). The result should be zero everywhere, except on the diagonal, where it is $T/2$:

In [ ]:
T = 1.0
basis = []
names = []
for n in range(1, 4):
    basis.append(lambda t, n=n: np.cos(2 * np.pi * n * t / T))
    names.append(f"cos {n}")
    basis.append(lambda t, n=n: np.sin(2 * np.pi * n * t / T))
    names.append(f"sin {n}")

gram = np.array([[inner_product(b1, b2, 0, T) for b2 in basis] for b1 in basis])

print("        " + " ".join(f"{name:>6}" for name in names))
for name, row in zip(names, gram):
    print(f"{name:>6}  " + " ".join(f"{value:6.2f}" for value in row))

Because they are orthogonal, we can use this basis to describe (almost) any signal on $[0, T]$ as a sum of sines and cosines. The weight of each basis function is given by the inner product of the signal with it (divided by $T/2$):

$$x(t) \approx \sum_{n} a_n \cos\left(\frac{2\pi n t}{T}\right) + b_n \sin\left(\frac{2\pi n t}{T}\right), \qquad a_n = \frac{2}{T}\langle x, \cos_n \rangle, \quad b_n = \frac{2}{T}\langle x, \sin_n \rangle$$

As an example, let's approximate a square wave with more and more sines:

In [ ]:
def square_wave(t):
    return np.sign(np.sin(2 * np.pi * t / T))


t = np.linspace(0, T, 1000)

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, n_sines in zip(axes, [1, 3, 15]):
    approximation = np.zeros_like(t)
    for n in range(1, n_sines + 1):
        sin_n = lambda t, n=n: np.sin(2 * np.pi * n * t / T)
        b_n = 2 / T * inner_product(square_wave, sin_n, 0, T)
        approximation += b_n * sin_n(t)

    ax.plot(t, square_wave(t), color="navy", label="square wave")
    ax.plot(t, approximation, color="firebrick", label="approximation")
    ax.set_title(f"{n_sines} sine(s)")
    ax.set_xlabel("t")
axes[0].legend()
plt.show()

## 3. The Fourier transform

What if we do not know in advance which frequencies are in a signal? We can compute the inner product of the signal with sinusoids of *every* frequency $\nu$. To keep track of both the sine and cosine parts (the phase) at once, we use complex exponentials, $e^{-2\pi i \nu t} = \cos(2\pi\nu t) - i \sin(2\pi\nu t)$. This is the **Fourier transform**:

$$F(\nu) = \langle f, e^{2\pi i \nu t} \rangle = \int_{-\infty}^{\infty} f(t)\, e^{-2\pi i \nu t}\, dt$$

$|F(\nu)|$ tells us how much of frequency $\nu$ is in the signal. Here we compute it numerically for a signal on a finite interval $[0, T]$, made of a 5 Hz and a 12 Hz sine:

In [ ]:
def signal(t):
    return np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 12 * t)


T = 2.0
freqs = np.linspace(0, 20, 401)  # the frequencies (in Hz) we want to test

F = np.array(
    [inner_product(signal, lambda t, nu=nu: np.exp(2j * np.pi * nu * t), 0, T) for nu in freqs]
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
t = np.linspace(0, T, 1000)
ax1.plot(t, signal(t), color="navy")
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("Amplitude")
ax2.plot(freqs, np.abs(F), color="firebrick")
ax2.set_xlabel("Frequency (Hz)")
ax2.set_ylabel("|F|")
plt.tight_layout()
plt.show()

## 4. The discrete Fourier transform

In practice, signals on a computer are not functions, but lists of samples $x[0], \dots, x[N-1]$, taken at a sampling rate $sr$. The integral becomes a sum, and we only look at $N$ frequencies, $\nu_k = k \cdot sr / N$. This is the **discrete Fourier transform** (DFT):

$$X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-2\pi i k n / N}, \qquad k = 0, \dots, N-1$$

We can write it in a few lines of code. In practice, we use the fast Fourier transform (FFT, `np.fft.fft`), which computes exactly the same thing, but much faster ($O(N \log N)$ instead of $O(N^2)$ operations).

In [ ]:
sr = 100  # sampling rate (samples per second)
N = 200  # number of samples (2 seconds)
x = signal(np.arange(N) / sr)  # sample the signal from before


def dft(x):
    N = len(x)
    n = np.arange(N)
    k = n.reshape(-1, 1)
    return np.exp(-2j * np.pi * k * n / N) @ x


X = dft(x)
print("Same result as np.fft.fft:", np.allclose(X, np.fft.fft(x)))

# For real signals, the second half of the spectrum mirrors the first half
freqs_dft = np.arange(N) * sr / N
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(freqs_dft[: N // 2], np.abs(X[: N // 2]), color="firebrick")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("|X|")
plt.show()

## 5. Interactive visualization: the spectrum of a piano tone

Let's use this to look at a real sound! Below is the recording of a piano playing an A4, and its (magnitude) spectrum computed with the DFT. A piano tone does not consist of a single sine wave: besides the fundamental frequency, it contains many **harmonics** (integer multiples of the fundamental).

In the interactive plot below:

- **Left click** on the spectrum adds a vertical line (a harmonic) at that frequency.
- **Right click** removes the most recently added line.
- The **Components** slider below the figure picks $N$ equally spaced components, starting at the fundamental frequency (the fundamental and its first $N-1$ overtones), and draws the $N$ lines. This replaces the lines you added by hand.

The audio player below the figure is updated every time the lines change, so you can listen to the additive synthesis of the harmonics you selected. The plot on the right compares the original piano signal (red) with the synthesized one (black).

In [ ]:
%matplotlib widget
import os

import matplotlib.pyplot as plt
from utils import InteractiveToneAnalyzer, load_audio, ipd, PATH

signal, sr = load_audio(os.path.join(PATH, "example_data", "a4_piano.wav"))

display(ipd.Audio(
            data=signal,
            rate=sr,
            normalize=True,
            element_id="original_signal",
        ))

plot = InteractiveToneAnalyzer(signal=signal, sample_rate=sr)

plot.show()